# Week 28, Day 5: Class vs instance attributes

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w028d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Dog:
    species = "Canis familiaris"
    legs = 4

    def __init__(self, name):
        self.name = name

rex, pip = Dog("Rex"), Dog("Pip")
pip.legs = 3
print(rex.species, rex.legs, pip.legs, Dog.legs)
print(vars(rex), vars(pip))

In [ ]:
class Pizza:
    PRICES = {"small": 750, "medium": 950, "large": 1150}
    TOPPING = 120

    def __init__(self, size, toppings=()):
        if size not in Pizza.PRICES:
            raise ValueError(f"no {size!r} size")
        self.size = size
        self.toppings = list(toppings)

    def price(self):
        return Pizza.PRICES[self.size] + Pizza.TOPPING * len(self.toppings)

order = [Pizza("large", ["olives", "peppers"]), Pizza("small")]
print([p.price() for p in order], sum(p.price() for p in order))

**Predict before you run:** what will this print?



In [ ]:
class Counter:
    total = 0

a, b = Counter(), Counter()
a.total += 1          # reads Counter.total (0), then sets a.total = 1
print(a.total, b.total, Counter.total)
Counter.total = 10
print(a.total, b.total)

In [ ]:
class Ticket:
    next_number = 1

    def __init__(self, holder):
        self.holder = holder
        self.number = Ticket.next_number
        Ticket.next_number += 1

tickets = [Ticket(n) for n in ["Ada", "Bo", "Cy"]]
print([(t.holder, t.number) for t in tickets], Ticket.next_number)

**Predict before you run:** what will this print?



In [ ]:
class Student:
    courses = []

    def __init__(self, name):
        self.name = name

    def enrol(self, course):
        self.courses.append(course)

ada, bo = Student("Ada"), Student("Bo")
ada.enrol("maths")
print(bo.courses)

In [ ]:
class Time:
    def __init__(self, hour, minute):
        if not (0 <= hour < 24 and 0 <= minute < 60):
            raise ValueError(f"no such time: {hour}:{minute}")
        self.hour = hour
        self.minute = minute

    @classmethod
    def from_text(cls, text):
        h, m = text.strip().split(":")
        return cls(int(h), int(m))

    @classmethod
    def from_minutes(cls, total):
        return cls(total // 60 % 24, total % 60)

    @staticmethod
    def is_valid_text(text):
        parts = text.strip().split(":")
        return len(parts) == 2 and all(p.isdigit() for p in parts) and int(parts[0]) < 24 and int(parts[1]) < 60

    def text(self):
        return f"{self.hour:02d}:{self.minute:02d}"

print(Time.from_text(" 9:05").text(), Time.from_minutes(1500).text())
print(Time.is_valid_text("23:59"), Time.is_valid_text("24:00"), Time.is_valid_text("noon"))

> **Key idea.** Class attributes, written in the class body, are shared by all instances: use them for constants, tables and counters. Reading `obj.x` looks in the instance, then the class; assigning `obj.x = ...` always sets it on the instance, which can shadow the class's. Change class attributes through the class, and never keep per-instance data in a mutable class attribute. `@classmethod` methods get the class as `cls` and make alternative constructors; `@staticmethod` methods get neither.

### Exercise: A table in the class

Write `Parcel(weight_kg, zone)` with a class attribute `RATES = {"local": 300, "national": 550, "international": 1400}` (pence for the first kilo) and `PER_EXTRA_KG = 150`. `__init__` raises `ValueError` for an unknown zone or a weight that isn't more than 0. `cost()` returns the zone's rate plus `PER_EXTRA_KG` for each **started** kilo after the first (so 2.1 kg pays for 2 extra).

In [ ]:
import math

class Parcel:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_costs():\n    \"costs\"\n    P = ns[\"Parcel\"]\n    assert P(1, \"local\").cost() == 300 and P(0.4, \"national\").cost() == 550\n    assert P(2.1, \"local\").cost() == 600 and P(3, \"international\").cost() == 1700\ndef test_shared():\n    \"the table is on the class\"\n    P = ns[\"Parcel\"]\n    assert \"RATES\" in vars(P) and \"PER_EXTRA_KG\" in vars(P)\n    p = P(1, \"local\")\n    assert \"RATES\" not in vars(p)\n    for args in [(1, \"moon\"), (0, \"local\"), (-2, \"local\")]:\n        try:\n            P(*args)\n            assert False, f\"Parcel{args} should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: Numbered members

Write `Member(name)` that gives each new member a `number` from a class counter `next_number`, starting at 1000, and also keeps a class attribute `count` of how many members have been made. Make sure the counters are changed on the class, not on the instance.

In [ ]:
class Member:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_numbers():\n    \"numbers and count\"\n    M = ns[\"Member\"]\n    start, made = M.next_number, M.count\n    a, b = M(\"Ada\"), M(\"Bo\")\n    assert (a.number, b.number) == (start, start + 1) and M.count == made + 2 and M.next_number == start + 2\n    assert \"next_number\" not in vars(a) and \"count\" not in vars(b)\ndef test_start():\n    \"starts at 1000\"\n    import ast\n    assert \"1000\" in source")

### Exercise: Who sees what?

Predict, then check.

In [ ]:
class Car:
    wheels = 4
    colour = "white"

a, b, c = Car(), Car(), Car()
b.colour = "red"
Car.colour = "blue"
c.wheels = c.wheels + 2
Car.wheels = 3

p = ___   # a.colour
q = ___   # b.colour
r = ___   # c.wheels
s = ___   # a.wheels
t = ___   # sorted(vars(c))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r, s, t) == (\"blue\", \"red\", 6, 3, [\"wheels\"])")

### Exercise: Another way in

Write `Money(pence)` (a whole number of pence, not negative, or `ValueError`) with a `text()` method giving `"£12.50"`, and a class method `from_text(text)` that accepts `"£12.50"`, `"12.5"`, `"12"` or `" £0.05 "` and returns a `Money`. Bad text raises `ValueError`. Use `cls(...)` to make the instance.

In [ ]:
import re

class Money:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    M = ns[\"Money\"]\n    assert M.from_text(\"\u00a312.50\").pence == 1250 and M.from_text(\"12.5\").pence == 1250\n    assert M.from_text(\" \u00a30.05 \").text() == \"\u00a30.05\" and M.from_text(\"12\").text() == \"\u00a312.00\"\n    assert isinstance(M.from_text(\"1\"), M) and M(123456).text() == \"\u00a31234.56\"\ndef test_bad():\n    \"bad input\"\n    M = ns[\"Money\"]\n    for t in [\"\", \"\u00a3\", \"12.345\", \"abc\", \"-3\"]:\n        try:\n            M.from_text(t)\n            assert False, f\"from_text({t!r}) should raise ValueError\"\n        except ValueError:\n            pass\n    for p in (-1, 2.5):\n        try:\n            M(p)\n            assert False, f\"Money({p}) should raise ValueError\"\n        except ValueError:\n            pass\ndef test_cls():\n    \"uses cls\"\n    assert \"cls(\" in source and \"@classmethod\" in source")

### Exercise: A helper that belongs here

Add a static method `is_valid_code(code)` to `Course`: True for codes of exactly three capital letters followed by three digits, like `"MTH101"`. Make `__init__` use it, raising `ValueError` for a bad code.

In [ ]:
import re

class Course:
    def __init__(self, code, title):
        self.code = code
        self.title = title

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    C = ns[\"Course\"]\n    assert C.is_valid_code(\"MTH101\") and not C.is_valid_code(\"mth101\") and not C.is_valid_code(\"MATH101\")\n    assert not C.is_valid_code(\"MTH10\") and not C.is_valid_code(\"MTH1011\")\n    assert C(\"PHY200\", \"Waves\").code == \"PHY200\"\n    try:\n        C(\"ab12\", \"Oops\")\n        assert False, \"a bad code should raise ValueError\"\n    except ValueError:\n        pass\n    assert \"@staticmethod\" in source")

### Bug hunt: Everybody's homework

Each `Pupil` should have its own homework list, but they all share one. Fix it.

In [ ]:
class Pupil:
    homework = []

    def __init__(self, name):
        self.name = name

    def set_task(self, task):
        self.homework.append(task)

ada, bo = Pupil("Ada"), Pupil("Bo")
ada.set_task("essay")
print(ada.homework, bo.homework)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"separate lists\"\n    expect_output(\"['essay'] []\")")

### Bug hunt: The counter that doesn't count

`Visitor.count` should be 3 after three visitors, but it stays 0. Fix it.

In [ ]:
class Visitor:
    count = 0

    def __init__(self, name):
        self.name = name
        self.count += 1

for n in ["Ada", "Bo", "Cy"]:
    Visitor(n)
print(Visitor.count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"counts\"\n    expect_output(\"3\")")

### Puzzle: An alternative constructor

Arrange the lines into a `Point` class with a class method that makes a point from text like `"3,4"`. One line is a trap.

(Put these lines in order on the website.)

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: event tickets

Write `Event(name, capacity)` and `Ticket`. Each event keeps its **own** list of tickets sold. Tickets are numbered per event, like `"GIG-001"`, `"GIG-002"`: the first three letters of the event's name in capitals, a hyphen, then a three-digit number counting from 1 for that event. A class attribute `Ticket.sold` counts tickets across **all** events.

- `event.sell(holder)` makes and returns a `Ticket` (with `code`, `holder` and `event` attributes), or raises `ValueError` when the event is full.
- `Ticket.from_code(events, code)` is a class method: given a list of events and a code, return the matching ticket, or raise `KeyError`.
- `event.left()` returns the places left.

In [ ]:
class Ticket:
    sold = 0
    ___

class Event:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_selling():\n    \"selling and numbering\"\n    E, T = ns[\"Event\"], ns[\"Ticket\"]\n    before = T.sold\n    gig, play = E(\"gig night\", 2), E(\"Play\", 5)\n    t1, t2, t3 = gig.sell(\"Ada\"), play.sell(\"Bo\"), gig.sell(\"Cy\")\n    assert (t1.code, t2.code, t3.code) == (\"GIG-001\", \"PLA-001\", \"GIG-002\")\n    assert t3.holder == \"Cy\" and t3.event is gig and gig.left() == 0 and play.left() == 4\n    assert T.sold == before + 3 and \"sold\" not in vars(t1)\n    try:\n        gig.sell(\"Di\")\n        assert False, \"a full event should raise ValueError\"\n    except ValueError:\n        pass\n    assert T.sold == before + 3\ndef test_lookup():\n    \"from_code\"\n    E, T = ns[\"Event\"], ns[\"Ticket\"]\n    a, b = E(\"Quiz\", 3), E(\"Dance\", 3)\n    a.sell(\"x\")\n    t = b.sell(\"y\")\n    assert T.from_code([a, b], \"DAN-001\") is t\n    try:\n        T.from_code([a, b], \"DAN-002\")\n        assert False, \"an unknown code should raise KeyError\"\n    except KeyError:\n        pass\n    assert \"@classmethod\" in source")

## Warm-up

`species` and `legs` are written in the class body, not in `__init__`, so they belong to the **class**. Every dog can read them. But `pip.legs = 3` didn't change the class: it gave Pip an attribute of its own, which hides the class one for Pip alone.

## Attributes that belong to the class

A name assigned directly in the class body (outside any method) is a **class attribute**. There's one copy, stored on the class, shared by every instance.

Use class attributes for things that are the **same for every instance**:

- constants: `MAX_SPEED = 70`, `VAT_RATE = 0.2`;
- lookup tables: a dictionary of prices by size;
- default values that instances may override.

Use instance attributes (set in `__init__`) for things that **differ** between instances: a name, a balance, a position.

Constants are written in CAPITALS by convention, like module-level constants. Inside methods you can reach a class attribute as `Pizza.PRICES` or as `self.PRICES` (the lookup finds it either way, as the next section explains).

## How Python finds an attribute

When you **read** `obj.name`, Python looks:

1. in the instance's own attributes (its `vars(obj)` dictionary);
2. if it isn't there, in the class;
3. (in Week 30: then in the classes it inherits from);
4. if it's nowhere, `AttributeError`.

When you **assign** `obj.name = value`, Python always sets it on the **instance**, never on the class. That's the source of **shadowing**: the instance attribute hides the class one from then on, for that instance only.

`a.total += 1` reads `total` (found on the class: 0), adds 1, and **assigns** `a.total = 1`, creating an instance attribute. `b` still sees the class's value, and so does everyone else after the class attribute changes, except `a`, whose own attribute now hides it.

Methods are class attributes too: `def deposit` in the class body puts a function called `deposit` on the class, and that's why every instance can find it.

## Counters and the shared-list trap

To change a class attribute, assign it **on the class**. A common use is counting instances, or handing out ids:

(Week 25's `next_id` lived in the data dictionary instead, so it could be saved to a file. A class attribute starts again every time the program runs.)

The other trap is a **mutable** class attribute. A list or dictionary in the class body is shared by every instance, so changing it through one instance changes it for all:

`self.courses.append(...)` doesn't **assign** an attribute, it changes the object found by lookup, which is the class's one list. Anything that differs per instance must be created in `__init__` (`self.courses = []`). Keep mutable class attributes for things that really are shared, and treat them as read-only (like `PRICES` above).

## Class methods and static methods

A normal method gets the **instance** as its first argument. Two decorators change that.

`@classmethod` makes a method that gets the **class** as its first argument, called `cls` by convention. Its main use is an **alternative constructor**: another way to make an instance, from a different kind of input.

- `Time.from_text("9:05")` is called on the class and returns a new `Time`. Using `cls(...)` rather than `Time(...)` means it will also work for subclasses (Week 30). Python's own types do the same: `dict.fromkeys(...)`, `datetime.fromisoformat(...)`, `int.from_bytes(...)`.
- `@staticmethod` makes a method that gets **neither** the instance nor the class: an ordinary function that lives in the class because it belongs with it. Use it sparingly; a plain function in the module is often just as good.

## Practice

## Quiz

## Daily challenge

## Before you go

For each class you've written this week, decide which attributes belong to every instance separately and which (if any) are the same for all of them. Tomorrow's Forge brings the week together in a bigger program built from several classes that work together.